# Bronze Layer - Raw to Bronze## Medallion Architecture - Bronze LayerThis notebook implements the **Bronze layer** of the Medallion architecture for ATLYS visa platform data.### Purpose- Copy data from `Atlys.raw` schema to `Atlys.bronze` schema- Bronze layer serves as an immutable copy of raw data- Minimal to no transformations- Preserves data lineage and auditability### Data Flow```Atlys.raw.* → Atlys.bronze.*```### Tables to Process1. raw_countries → bronze_countries2. raw_visa_types → bronze_visa_types3. raw_airports → bronze_airports4. raw_users → bronze_users5. raw_passports → bronze_passports6. raw_applications → bronze_applications7. raw_payments → bronze_payments8. raw_documents → bronze_documents9. raw_application_events → bronze_application_events10. raw_reviews → bronze_reviews11. raw_support_tickets → bronze_support_tickets

In [ ]:
from pyspark.sql import SparkSessionfrom datetime import datetimespark = SparkSession.builder.getOrCreate()SOURCE_CATALOG = "Atlys"SOURCE_SCHEMA = "raw"TARGET_CATALOG = "Atlys"TARGET_SCHEMA = "bronze"TABLES = [    "countries",    "visa_types",    "airports",    "users",    "passports",    "applications",    "payments",    "documents",    "application_events",    "reviews",    "support_tickets"]print(f"Source: {SOURCE_CATALOG}.{SOURCE_SCHEMA}")print(f"Target: {TARGET_CATALOG}.{TARGET_SCHEMA}")print(f"Tables to process: {len(TABLES)}")

In [ ]:
print("Creating Bronze schema...")if TARGET_CATALOG not in ["main", "hive_metastore"]:    spark.sql(f"CREATE CATALOG IF NOT EXISTS {TARGET_CATALOG}")    print(f"✓ Catalog '{TARGET_CATALOG}' ready")spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")print(f"✓ Schema '{TARGET_CATALOG}.{TARGET_SCHEMA}' created")schemas = spark.sql(f"SHOW SCHEMAS IN {TARGET_CATALOG}").collect()schema_names = [row.databaseName for row in schemas]print(f"\nAvailable schemas in {TARGET_CATALOG}: {schema_names}")

In [ ]:
def copy_raw_to_bronze(table_name: str, mode: str = "overwrite"):    """    Copy data from raw to bronze layer with minimal transformations.        Args:        table_name: Name of the table (without schema prefix)        mode: Write mode - 'overwrite' or 'append'    """    source_table = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.raw_{table_name}"    target_table = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.bronze_{table_name}"        try:        print(f"\n📦 Processing: {table_name}")        print(f"   Source: {source_table}")        print(f"   Target: {target_table}")                # Read from raw layer        df = spark.table(source_table)        row_count = df.count()        print(f"   Rows: {row_count:,}")                # Add bronze layer metadata columns        from pyspark.sql.functions import current_timestamp, lit                df_bronze = df \            .withColumn("bronze_ingestion_timestamp", current_timestamp()) \            .withColumn("bronze_source_table", lit(source_table))                # Write to bronze layer        df_bronze.write \            .format("delta") \            .mode(mode) \            .option("overwriteSchema", "true") \            .saveAsTable(target_table)                print(f"   ✓ Success: Copied {row_count:,} rows to {target_table}")        return True, row_count            except Exception as e:        print(f"   ✗ Error: {str(e)}")        return False, 0print("✓ Bronze copy function defined")

In [ ]:
print("=" * 80)print("BRONZE LAYER INGESTION - Raw to Bronze")print("=" * 80)results = []total_rows = 0for table in TABLES:    success, row_count = copy_raw_to_bronze(table, mode="overwrite")    results.append({"table": table, "success": success, "rows": row_count})    total_rows += row_countprint("\n" + "=" * 80)print("BRONZE LAYER INGESTION COMPLETE")print("=" * 80)successful = sum(1 for r in results if r["success"])failed = len(results) - successfulprint(f"\n📊 Summary:")print(f"   Total tables: {len(results)}")print(f"   Successful: {successful}")print(f"   Failed: {failed}")print(f"   Total rows copied: {total_rows:,}")import pandas as pddf_results = pd.DataFrame(results)display(df_results)

In [ ]:
-- Verify all bronze tables existSHOW TABLES IN Atlys.bronze

In [ ]:
-- Sample data from bronze users tableSELECTuser_id,first_name,last_name,email,country,signup_date,bronze_ingestion_timestamp,bronze_source_tableFROM Atlys.bronze.bronze_usersLIMIT 10